In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from pathlib import Path
import json
sys.path.append('../')
from data_provider.data_loader import load_dataset


### Experiment Configuration

In [ ]:
# Data and Embedding Configurations
DATA_FOLDER = '../data'
DATASET_NAME = 'experiment3'
EMBEDDINGS_FOLDER = DATA_FOLDER + '/embeddings'

EMBEDDING_OPTIONS = {
    'glove': {
        'file_path': EMBEDDINGS_FOLDER + '/GloVe' + '/wiki_giga_2024_300_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt',
        'embedding_dim': 300,
        'skip_first_line': False,
    },
    'word2vec': {
        'file_path': EMBEDDINGS_FOLDER + '/word2vec' + '/GoogleNews-vectors-negative300.txt',
        'embedding_dim': 300,
        'skip_first_line': True,
    },
    'fasttext': {
        'file_path': EMBEDDINGS_FOLDER + '/fast_text' + '/crawl-300d-2M-subword' + '/crawl-300d-2M-subword.vec',
        'embedding_dim': 300,
        'skip_first_line': True,
    },
}


EMBEDDING_SOURCE = 'fasttext' # Change this to compare embeddings: 'glove', 'word2vec', or 'fasttext'.
EMBEDDING_CONFIG = EMBEDDING_OPTIONS[EMBEDDING_SOURCE]
EMBEDDING_PATH = EMBEDDING_CONFIG['file_path']
EMBEDDING_DIM = EMBEDDING_CONFIG['embedding_dim']

# Date Preprocessing
MAX_VOCAB_SIZE = 30000
MAX_SEQUENCE_LENGTH = 256
PAD_IDX = 0
TEST_SIZE = 0.2
VAL_SIZE = 0.1
RANDOM_SEED = 2026
REBUILD_EMBEDDING_MATRIX = True  # Set to True to rebuild the embedding matrix, False to load from file if it exists

# Model Hyperparameters
HIDDEN_DIM = 128
NUM_LAYERS = 1
BIDIRECTIONAL = True
DROPOUT = 0.3
FREEZE_EMBEDDINGS = False

# Training and Optimization
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3

# Early Stopping
PATIENCE=3
MIN_DELTA=0.001

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cpu')

#### Load Dataset

In [ ]:
df = load_dataset(DATA_FOLDER, DATASET_NAME)

# Check the column names to ensure features and labels are loaded correctly
if isinstance(df, dict):
    print(df['test'].head())
else:
    print(df.head())

                                                text  label
0  Phones\n\nModern humans today are always on th...      0
1  This essay will explain if drivers should or s...      0
2  Driving while the use of cellular devices\n\nT...      0
3  Phones & Driving\n\nDrivers should not be able...      0
4  Cell Phone Operation While Driving\n\nThe abil...      0


#### Split Dataset

In [ ]:
# Dataset is not randomly ordered, so we will shuffle it to ensure a representative split between training and testing sets

if isinstance(df, dict):
    train_val_df = df['train']
    test_df = df['test']
else:
    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=df['label'],
        shuffle=True
    )
train_df, val_df = train_test_split(
    train_val_df,
    test_size=(VAL_SIZE / (1.0 - TEST_SIZE)),
    random_state=RANDOM_SEED,
    stratify=train_val_df['label'],
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train set size: {len(train_df)}')
print(f'Validation set size: {len(val_df)}')
print(f'Test set size: {len(test_df)}')


Train set size: 31407
Validation set size: 4487
Test set size: 8974


#### Tokenization and Vocabulary

In [191]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<UNK>')
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE)
print(f'Vocabulary size: {vocab_size}')


Vocabulary size: 30000


#### Load Pretrained Embeddings

In [ ]:
def get_token_candidates(token):
    """
    Generate different case variations of the token
    """
    return [token, token.lower(), token.capitalize(), token.upper()]


def get_pretrained_vector(pretrained_vectors, token):
    """
    Check different case variations of the token to find a match in the pretrained embeddings
    """
    for candidate in get_token_candidates(token):
        if candidate in pretrained_vectors:
            return pretrained_vectors[candidate]
        
    return None


def load_embedding_lookup(word_to_idx, embedding_path, embedding_dim, skip_first_line=False):
    """Load pretrained embeddings into a dictionary"""
    target_tokens = set()

    for word in word_to_idx:
        for candidate in get_token_candidates(word):
            target_tokens.add(candidate)

    pretrained_vectors = {}

    with open(embedding_path, 'r', encoding='utf-8', errors='ignore') as embedding_file:
        for line_number, line in enumerate(tqdm(embedding_file)):
            if skip_first_line and line_number == 0:
                continue
            
            # Split the line into token and vector components
            parts = line.rstrip().split()
            token = parts[0]
            
            if len(parts) != embedding_dim + 1 or token not in target_tokens:
                continue

            pretrained_vectors[token] = np.asarray(parts[1:], dtype=np.float32)

    return pretrained_vectors


def build_embedding_matrix(word_to_idx, embedding_path, embedding_dim, skip_first_line=False):
    """
    Build the embedding matrix
    """
    vocab_size = min(len(word_to_idx) + 1, MAX_VOCAB_SIZE)

    pretrained_vectors = load_embedding_lookup(
        word_to_idx,
        embedding_path,
        embedding_dim,
        skip_first_line=skip_first_line,
    )

    # Initialize embedding matrix with random values
    embedding_matrix = np.random.normal(
        size=(vocab_size, embedding_dim),
    ).astype(np.float32)

    # Set the embedding for the padding token to zeros
    embedding_matrix[PAD_IDX] = np.zeros(embedding_dim, dtype=np.float32)

    hits = 0

    # Fill the embedding matrix with the pretrained vectors
    for word, idx in tqdm(word_to_idx.items()):
        if idx == PAD_IDX or idx >= vocab_size:
            continue
        vector = get_pretrained_vector(pretrained_vectors, word)
        if vector is not None:
            embedding_matrix[idx] = vector
            hits += 1

    coverage = hits / max(1, len(word_to_idx) - 1)
    print(f'Loaded embeddings from {embedding_path}')
    print(f'Embedding coverage: {coverage:.2%}')
    
    return embedding_matrix

if REBUILD_EMBEDDING_MATRIX:
    embedding_matrix = build_embedding_matrix(
        tokenizer.word_index,
        EMBEDDING_PATH,
        EMBEDDING_DIM,
        skip_first_line=EMBEDDING_CONFIG['skip_first_line'],
    )

    # Save the embedding matrix for future use
    np.save('embedding_matrix_fasttext.npy', embedding_matrix)
else:
    # Load the embedding matrix
    embedding_matrix = np.load(f'embedding_matrix_{EMBEDDING_SOURCE}.npy')


2000001it [00:17, 111479.35it/s]
100%|██████████| 76125/76125 [00:00<00:00, 2404682.91it/s]

Loaded embeddings from ../data/embeddings/fast_text/crawl-300d-2M-subword/crawl-300d-2M-subword.vec
Embedding coverage: 33.45%


#### Dataset and DataLoaders

In [ ]:
class TextSequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels.to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]

train_dataset = TextSequenceDataset(X_train, train_df['label'])
val_dataset = TextSequenceDataset(X_val, val_df['label'])
test_dataset = TextSequenceDataset(X_test, test_df['label'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)



#### Define LSTM Model

In [194]:
class LSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_layers=1, bidirectional=True, dropout=0.3, freeze_embeddings=False):
        super().__init__()

        embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)
        self.embedding = nn.Embedding.from_pretrained(embedding_tensor, freeze=freeze_embeddings,padding_idx=PAD_IDX)

        self.lstm = nn.LSTM(
            input_size=embedding_tensor.shape[1],
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(output_dim, 1)
        self.bidirectional = bidirectional

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output, (hidden_state, cell_state) = self.lstm(embeddings)

        if self.bidirectional:
            final_hidden = torch.cat((hidden_state[-2], hidden_state[-1]), dim=1)
        else:
            final_hidden = hidden_state[-1]

        logits = self.classifier(self.dropout(final_hidden))
        
        return logits.squeeze(1)

model = LSTM(
    embedding_matrix=embedding_matrix,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    freeze_embeddings=FREEZE_EMBEDDINGS,
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Verify the model is defined correctly
model


LSTM(
  (embedding): Embedding(30000, 300, padding_idx=0)
  (lstm): LSTM(300, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)

#### Training and Evaluation

In [ ]:
def run_epoch(model, data_loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []
    y_scores = []

    def process_batch(input_ids, labels):
        logits = model(input_ids)
        loss = criterion(logits, labels)

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        return loss, logits, labels
    
    if is_training:
        for input_ids, labels in tqdm(data_loader):
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            val_loss, logits, labels = process_batch(input_ids, labels)

            total_loss += val_loss.item() * input_ids.size(0)
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).long()

            y_true.extend(labels.detach().cpu().numpy().astype(int))
            y_pred.extend(predictions.detach().cpu().numpy())
            y_scores.extend(probabilities.detach().cpu().numpy())
    else:
        with torch.no_grad():
            for input_ids, labels, in tqdm(data_loader):
                input_ids = input_ids.to(DEVICE)
                labels = labels.to(DEVICE)

                val_loss , logits, labels = process_batch(input_ids, labels)

                total_loss += val_loss.item() * input_ids.size(0)
                probabilities = torch.sigmoid(logits)
                predictions = (probabilities >= 0.5).long()
                
                y_true.extend(labels.detach().cpu().numpy().astype(int))
                y_pred.extend(predictions.detach().cpu().numpy())
                y_scores.extend(probabilities.detach().cpu().numpy())

    average_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(y_true, y_pred)
    
    return average_loss, accuracy, y_true, y_pred, y_scores


best_val_accuracy = 0.0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy, train_true, train_pred, train_scores = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_accuracy, val_true, val_pred, val_scores = run_epoch(model, val_loader, criterion)

    print(
        f'Epoch {epoch + 1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}'
    )

    if val_accuracy > best_val_accuracy + MIN_DELTA:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_loss, test_accuracy, y_true, y_pred, y_scores = run_epoch(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))
print(f'Test ROC-AUC: {roc_auc_score(y_true, y_scores):.4f}')

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI']).plot(cmap='Blues')
plt.title('LSTM Confusion Matrix')
plt.show()

results_dir = Path('../results') / f'lstm__{DATASET_NAME}__{EMBEDDING_SOURCE}'
results_dir.mkdir(parents=True, exist_ok=True)

cm = confusion_matrix(y_true, y_pred)
confusion_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI'])
confusion_display.plot(cmap='Blues')
plt.title('LSTM Confusion Matrix')
plt.show()

pd.DataFrame(cm, index=['Human', 'AI'], columns=['Predicted Human', 'Predicted AI']).to_csv(results_dir / 'confusion_matrix.csv')
confusion_display.figure_.savefig(results_dir / 'confusion_matrix.png', bbox_inches='tight')

classification_report_dict = classification_report(
    y_true,
    y_pred,
    target_names=['Human', 'AI'],
    output_dict=True,
)

metrics = {
    'dataset_name': DATASET_NAME,
    'embedding_source': EMBEDDING_SOURCE,
    'test_loss': float(test_loss),
    'test_accuracy': float(test_accuracy),
    'test_roc_auc': float(roc_auc_score(y_true, y_scores)),
    'num_test_samples': int(len(y_true)),
}

with (results_dir / 'metrics.json').open('w', encoding='utf-8') as metrics_file:
    json.dump(metrics, metrics_file, indent=2)

with (results_dir / 'classification_report.json').open('w', encoding='utf-8') as report_file:
    json.dump(classification_report_dict, report_file, indent=2)

if DATASET_NAME == 'experiment3':
    source_results = []

    for source_column, source_subset in test_df.groupby('source'):
        source_texts = source_subset['text'].astype(str).tolist()
        source_sequences = tokenizer.texts_to_sequences(source_texts)
        source_sequences = pad_sequences(
            source_sequences,
            maxlen=MAX_SEQUENCE_LENGTH,
            padding='post',
            truncating='post',
        )
        source_tensor = torch.tensor(source_sequences, dtype=torch.long, device=DEVICE)

        with torch.no_grad():
            source_probabilities = torch.sigmoid(model(source_tensor)).cpu().numpy()

        source_predictions = (source_probabilities >= 0.5).astype(int)
        source_results.append({
            'source_model': source_column,
            'samples': len(source_texts),
            'detected_as_ai_rate': float(source_predictions.mean()),
        })

    source_results_df = pd.DataFrame(source_results)
    
    print('Per-source detection rates on multi_model_detection.csv:')
    print(source_results_df)
    source_results_df.to_csv(results_dir / 'source_detection_rates.csv', index=False)

print(f'Saved LSTM results to {results_dir}')

100%|██████████| 71/71 [00:04<00:00, 14.55it/s]


Epoch 1/100 | Train Loss: 0.2234 | Train Acc: 0.9105 | Val Loss: 0.0679 | Val Acc: 0.9806


100%|██████████| 71/71 [00:04<00:00, 14.57it/s]


Epoch 2/100 | Train Loss: 0.0684 | Train Acc: 0.9803 | Val Loss: 0.0815 | Val Acc: 0.9708


100%|██████████| 71/71 [00:04<00:00, 14.57it/s]


...

Epoch 9/100 | Train Loss: 0.0087 | Train Acc: 0.9977 | Val Loss: 0.0398 | Val Acc: 0.9882


100%|██████████| 71/71 [00:04<00:00, 14.46it/s]


Epoch 10/100 | Train Loss: 0.0033 | Train Acc: 0.9992 | Val Loss: 0.0274 | Val Acc: 0.9933


100%|██████████| 141/141 [00:10<00:00, 13.84it/s]

Test Loss: 0.0409
Test Accuracy: 0.9901
              precision    recall  f1-score   support

       Human       0.99      0.99      0.99      5474
          AI       0.99      0.99      0.99      3500

    accuracy                           0.99      8974
   macro avg       0.99      0.99      0.99      8974
weighted avg       0.99      0.99      0.99      8974



#### Predicting Custom Text Data

In [196]:
def predict_custom_texts(model, texts, tokenizer, max_length):
    if isinstance(texts, str):
        texts = [texts]

    sequences = tokenizer.texts_to_sequences(texts)
    sequences = pad_sequences(
        sequences,
        maxlen=max_length,
        padding='post',
        truncating='post',
    )
    input_tensor = torch.tensor(sequences, dtype=torch.long, device=DEVICE)

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(input_tensor)).cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)
    results = pd.DataFrame({
        'text': texts,
        'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
        'ai_probability': probabilities,
    })

    print(results)

    return results

#### Test LSTM with Custom Inputs

In [198]:
custom_texts = [
    "This essay argues that social media can improve civic participation when paired with strong moderation policies.",
    "In conclusion, the provided evidence clearly demonstrates a multifaceted and highly structured perspective on the topic.",
    "This isn't working very well, as it seems to just be predicting AI for longer messages and Human for shorter messages. At least it's with a lower probability.",
    "Texting while driving is a combination of all three types of distractions. It is the most dangerous distraction because it takes your eyes, hands, and mind off the road. When you text while driving, you are 23 times more likely to get into a crash than if you were not texting.",
]

predict_custom_texts(model, custom_texts, tokenizer, MAX_SEQUENCE_LENGTH)

                                                text predicted_label  \
0  This essay argues that social media can improv...              AI   
1  In conclusion, the provided evidence clearly d...              AI   
2  This isn't working very well, as it seems to j...              AI   
3  Texting while driving is a combination of all ...           Human   

   ai_probability  
0        0.999636  
1        0.998111  
2        0.949269  
3        0.000370  


,text,predicted_label,ai_probability
0,This essay argues that social media can improv...,AI,0.999636
1,"In conclusion, the provided evidence clearly d...",AI,0.998111
2,"This isn't working very well, as it seems to j...",AI,0.949269
3,Texting while driving is a combination of all ...,Human,0.000370


#### Save LSTM Artifacts

In [204]:
from pathlib import Path
import json
import pickle

SAVE_DIR = Path('../artifacts/lstm')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), SAVE_DIR / 'weights.pt')
np.save(SAVE_DIR / 'embedding_matrix.npy', embedding_matrix)

with (SAVE_DIR / 'tokenizer.pkl').open('wb') as tokenizer_file:
    pickle.dump(tokenizer, tokenizer_file)

lstm_config = {
    'dataset_name': DATASET_NAME,
    'embedding_source': EMBEDDING_SOURCE,
    'embedding_path': EMBEDDING_PATH,
    'embedding_dim': EMBEDDING_DIM,
    'max_vocab_size': MAX_VOCAB_SIZE,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'bidirectional': BIDIRECTIONAL,
    'dropout': DROPOUT,
    'freeze_embeddings': FREEZE_EMBEDDINGS,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'patience': PATIENCE,
    'min_delta': MIN_DELTA,
}

with (SAVE_DIR / 'config.json').open('w', encoding='utf-8') as config_file:
    json.dump(lstm_config, config_file, indent=2)

print(f'Saved LSTM artifacts to {SAVE_DIR}')


Saved LSTM artifacts to ../artifacts/lstm


#### Reload LSTM Artifacts

In [ ]:
from pathlib import Path
import pickle

RELOAD_DIR = Path('../artifacts/lstm')

with (RELOAD_DIR / 'tokenizer.pkl').open('rb') as tokenizer_file:
    reloaded_tokenizer = pickle.load(tokenizer_file)

reloaded_embedding_matrix = np.load(RELOAD_DIR / 'embedding_matrix.npy')
reloaded_lstm_model = LSTM(
    embedding_matrix=reloaded_embedding_matrix,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    freeze_embeddings=FREEZE_EMBEDDINGS,
).to(DEVICE)
reloaded_lstm_model.load_state_dict(torch.load(RELOAD_DIR / 'weights.pt', map_location=DEVICE))
reloaded_lstm_model.eval()

print('Reloaded LSTM artifacts.')
